# 📊 NIFTY 50 Daily Data

Utility to fetch, initialize, and incrementally update **NIFTY 50 OHLCV data**  
from Yahoo Finance and store it as a local CSV file.

---

## 📁 Dataset Structure

| Column   | Type    | Description                                                                 |
|:---------|:--------|:----------------------------------------------------------------------------|
| Date     | string  | Trading session date in YYYY-MM-DD format. Data is sorted newest → oldest. |
| Open     | float   | Price at which NIFTY 50 opened at the start of the session.                |
| High     | float   | Highest price reached during the session.                                  |
| Low      | float   | Lowest price reached during the session.                                   |
| Close    | float   | Closing price at the end of the session (adjusted).                        |
| Volume   | integer | Total traded volume during the session.                                    |

---

## 📈 Derived Columns

| Column     | Type   | Formula                                              | Description                                                                 |
|:-----------|:-------|:-----------------------------------------------------|:----------------------------------------------------------------------------|
| Change_%   | float  | (Close - prev_Close) / prev_Close × 100               | Daily % change in closing price. Positive = up, negative = down. First row = 0. |
| Trend      | string | Based on Change_%                                    | Direction label: Positive, Negative, Neutral.                               |
| Range      | float  | High - Low                                           | Intraday volatility in absolute points.                                     |
| Range_%    | float  | (High - Low) / Low × 100                             | Range as % of low price (normalized volatility).                            |
| Gap        | float  | Open_today - Close_yesterday                         | Opening gap vs previous close. Positive = gap up, negative = gap down.      |
| Gap_%      | float  | (Open_today - Close_yesterday) / Close_yesterday × 100| Gap as % of previous close. First row = 0.                                  |

---

## ⚙️ Notes

- Core columns (Date, Open, High, Low, Close, Volume) come directly from Yahoo Finance.
- Data is sorted in **descending order (latest first)**.
- Missing values (NaN) are replaced with `0`.
- First row:
  - Change_% = 0  
  - Gap = 0  
  - Gap_% = 0  

---

## ⚙️ Behaviour & Edge Cases

- Derived columns (`Change_%`, `Trend`, `Range`, `Range_%`, `Gap`, `Gap_%`) are **recomputed from scratch** on every update to ensure the boundary row between old and new data is always correct.
- `Trend` is classified **before** NaN replacement — so the first row correctly gets `Neutral` rather than being misclassified.
- `Change_%`, `Gap`, and `Gap_%` for the **first ever row** are `0` because there is no prior session to reference.

---


## 📌 Use Cases

| Use Case | Description |
|:---------|:------------|
| 📊 Technical analysis | Feed into indicators like RSI, MACD, Bollinger Bands |
| 🤖 Machine learning | Use as feature input for price prediction models |
| 🔁 Backtesting | Replay historical sessions to test trading strategies |
| 📈 Dashboards | Plug into Streamlit, Power BI, or Grafana for visualisation |

In [1]:
# Import Libraries

import os
from datetime import datetime, timedelta

import pandas as pd
import yfinance as yf

In [2]:
# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
 
TICKER = "^NSEI"
 
CSV_COLUMNS = [
    "Date", "Open", "High", "Low", "Close", "Volume",
    "Change_%", "Trend",
    "Range", "Range_%",
    "Gap", "Gap_%",
]
 
# Derived columns that must be dropped and fully recomputed on every update
_DERIVED_COLS = ["Change_%", "Trend", "Range", "Range_%", "Gap", "Gap_%"]

In [3]:
# ---------------------------------------------------------------------------
# Private helpers
# ---------------------------------------------------------------------------
 
def _clean_raw_df(raw: pd.DataFrame) -> pd.DataFrame:
    """
    Normalise the raw DataFrame returned by ``yfinance.download``.
 
    Responsibilities
    ----------------
    * Flatten multi-level column index that yfinance sometimes produces,
      e.g. ``('Close', '^NSEI')`` → ``'Close'``.
    * Move the DatetimeIndex into a plain ``Date`` column.
    * Drop any spurious ``Price`` column injected by yfinance.
    * Retain only the six raw OHLCV columns.
    * Format ``Date`` as a ``YYYY-MM-DD`` string for CSV readability.
 
    Parameters
    ----------
    raw : pd.DataFrame
        DataFrame as returned by ``yf.download()``.
 
    Returns
    -------
    pd.DataFrame
        Cleaned DataFrame with columns:
        ``['Date', 'Open', 'High', 'Low', 'Close', 'Volume']``.
 
    Raises
    ------
    ValueError
        If any required OHLCV column is missing after cleaning.
    """
    df = raw.copy()
    df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]
    df.reset_index(inplace=True)
    df = df.loc[:, ~df.columns.str.contains("Price", case=False)]
 
    required = ["Date", "Open", "High", "Low", "Close", "Volume"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing expected columns after cleaning: {missing}")
 
    df = df[required].reset_index(drop=True)
    df["Date"] = pd.to_datetime(df["Date"]).dt.strftime("%Y-%m-%d")
    return df


In [4]:
def _add_change(df: pd.DataFrame) -> pd.DataFrame:
    """
    Append ``Change_%`` — daily percentage change in closing price.
 
    Formula::
 
        Change_% = (Close_today − Close_yesterday) / Close_yesterday × 100
 
    The first row has no prior close; its NaN is replaced with 0 by the
    downstream ``_fill_na`` step.
 
    Parameters
    ----------
    df : pd.DataFrame
        Must contain a ``Close`` column, sorted ascending by Date.
 
    Returns
    -------
    pd.DataFrame
        Input DataFrame with ``Change_%`` appended (float, 4 d.p.).
    """
    df = df.copy()
    df["Change_%"] = (df["Close"].pct_change() * 100).round(4)
    return df
 
 
def _add_trend(df: pd.DataFrame) -> pd.DataFrame:
    """
    Append ``Trend`` — direction label derived from ``Change_%``.
 
    Classification rules
    --------------------
    +----------------------+-------------+
    | Condition            | Label       |
    +======================+=============+
    | Change_% > 0         | Positive    |
    +----------------------+-------------+
    | Change_% < 0         | Negative    |
    +----------------------+-------------+
    | Change_% == 0 or NaN | Neutral     |
    +----------------------+-------------+
 
    Parameters
    ----------
    df : pd.DataFrame
        Must contain a ``Change_%`` column.
 
    Returns
    -------
    pd.DataFrame
        Input DataFrame with ``Trend`` appended (str).
    """
    df = df.copy()
 
    def _classify(val):
        if pd.isna(val) or val == 0:
            return "Neutral"
        return "Positive" if val > 0 else "Negative"
 
    df["Trend"] = df["Change_%"].apply(_classify)
    return df
 
 
def _add_range(df: pd.DataFrame) -> pd.DataFrame:
    """
    Append ``Range`` and ``Range_%`` — intraday price spread.
 
    ``Range`` (points)
        Absolute point distance between the session's high and low::
 
            Range = High − Low
 
    ``Range_%`` (percent)
        Range expressed as a percentage of the day's Low, showing how
        wide the session was relative to its floor::
 
            Range_% = (High − Low) / Low × 100
 
    Both values are rounded to 2 decimal places.
 
    Parameters
    ----------
    df : pd.DataFrame
        Must contain ``High`` and ``Low`` columns.
 
    Returns
    -------
    pd.DataFrame
        Input DataFrame with ``Range`` and ``Range_%`` appended.
 
    """
    df = df.copy()
    df["Range"]   = (df["High"] - df["Low"]).round(2)
    df["Range_%"] = ((df["High"] - df["Low"]) / df["Low"] * 100).round(2)
    return df
 
 
def _add_gap(df: pd.DataFrame) -> pd.DataFrame:
    """
    Append ``Gap`` and ``Gap_%`` — overnight price gap.
 
    A gap measures how far today's open jumped from yesterday's close.
    Positive = market opened higher; negative = market opened lower.
 
    ``Gap`` (points)::
 
        Gap = Open_today − Close_yesterday
 
    ``Gap_%`` (percent)::
 
        Gap_% = (Open_today − Close_yesterday) / Close_yesterday × 100
 
    The first trading day has no prior close; NaN is replaced with 0 by
    the downstream ``_fill_na`` step.  Both values are rounded to 2 d.p.
 
    Parameters
    ----------
    df : pd.DataFrame
        Must contain ``Open`` and ``Close`` columns, sorted ascending by
        Date so that ``shift(1)`` picks up the correct previous session.
 
    Returns
    -------
    pd.DataFrame
        Input DataFrame with ``Gap`` and ``Gap_%`` appended.
    """
    df = df.copy()
    prev_close  = df["Close"].shift(1)
    df["Gap"]   = (df["Open"] - prev_close).round(2)
    df["Gap_%"] = ((df["Open"] - prev_close) / prev_close * 100).round(2)
    return df
 

In [5]:
def _fill_na(df: pd.DataFrame) -> pd.DataFrame:
    """
    Replace all NaN / missing numeric values with ``0``.
 
    NaN values arise naturally in the first row of any diff / pct_change
    calculation (``Change_%``, ``Gap``, ``Gap_%``).  Replacing them with
    0 keeps the CSV clean and avoids type-casting surprises in downstream
    tools.  The ``Trend`` string column already receives ``'Neutral'`` for
    its first row and is unaffected.
 
    Parameters
    ----------
    df : pd.DataFrame
        Fully-derived DataFrame before saving.
 
    Returns
    -------
    pd.DataFrame
        Same DataFrame with all NaN in numeric columns replaced by 0.
    """
    df = df.copy()
    numeric_cols = df.select_dtypes(include="number").columns
    df[numeric_cols] = df[numeric_cols].fillna(0)
    return df
 
 
def _sort_descending(df: pd.DataFrame) -> pd.DataFrame:
    """
    Sort the DataFrame by ``Date`` descending — newest session at row 0.
 
    This makes the CSV immediately useful: open it and the latest data is
    visible without scrolling or sorting.
 
    Parameters
    ----------
    df : pd.DataFrame
        Must contain a ``Date`` column.
 
    Returns
    -------
    pd.DataFrame
        Same data, newest-first, with a clean 0-based integer index.
    """
    df = df.copy()
    df.sort_values("Date", ascending=False, inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df
 
 
def _build_derived(df: pd.DataFrame) -> pd.DataFrame:
    """
    Run the full derived-column pipeline on a base OHLCV DataFrame.
 
    Pipeline (order is mandatory)
    ------------------------------
    1. ``_add_change``      — requires Close; ascending sort
    2. ``_add_trend``       — requires Change_%
    3. ``_add_range``       — requires High, Low
    4. ``_add_gap``         — requires Open, Close; ascending sort
    5. ``_fill_na``         — NaN → 0
    6. ``_sort_descending`` — newest row first
    7. Column reorder       — enforces ``CSV_COLUMNS`` order
 
    Parameters
    ----------
    df : pd.DataFrame
        Base OHLCV DataFrame sorted **ascending** by Date, with no
        derived columns present.
 
    Returns
    -------
    pd.DataFrame
        Fully enriched DataFrame matching ``CSV_COLUMNS``, sorted
        descending, with no NaN values.
    """
    df = _add_change(df)
    df = _add_trend(df)
    df = _add_range(df)
    df = _add_gap(df)
    df = _fill_na(df)
    df = _sort_descending(df)
    return df[CSV_COLUMNS]

In [6]:
# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------
 
def update_nifty_csv(filepath: str = "nifty_50.csv") -> pd.DataFrame:
    """
    Create or incrementally update a local CSV with NIFTY 50 daily data.
 
    Behaviour
    ---------
    **File does not exist**
        Downloads the complete available history from Yahoo Finance,
        computes all derived columns, sorts newest-first, replaces NaN
        with 0, and writes a fresh CSV with the header defined by
        ``CSV_COLUMNS``.
 
    **File exists**
        Reads the most-recent date already stored, fetches only sessions
        from the following day through today, merges the new raw rows with
        the existing base OHLCV data, then **recomputes all derived columns
        from scratch** so boundary rows (Gap, Change_%) are always correct.
        The result overwrites the existing file.
 
    Output columns
    --------------
    ``Date, Open, High, Low, Close, Volume,``
    ``Change_%, Trend,``
    ``Range, Range_%,``
    ``Gap, Gap_%``
 
    Parameters
    ----------
    filepath : str, optional
        Path to the CSV file.  Defaults to ``"nifty_50.csv"`` in the
        current working directory.
 
    Returns
    -------
    pd.DataFrame
        Complete, up-to-date DataFrame persisted to ``filepath``.
        Sorted descending by Date, no NaN values.
 
    Raises
    ------
    ValueError
        If the existing CSV is empty or missing the ``Date`` column.
    RuntimeError
        If Yahoo Finance returns no data for a full-history request.
    """
 
    file_exists = os.path.isfile(filepath)
 
    # ------------------------------------------------------------------
    # Branch A: file does not exist → full download
    # ------------------------------------------------------------------
    if not file_exists:
        print(f"[INFO] '{filepath}' not found. Downloading full NIFTY 50 history…")
 
        raw = yf.download(TICKER, period="max", interval="1d", progress=False)
 
        if raw.empty:
            raise RuntimeError(
                f"Yahoo Finance returned no data for ticker '{TICKER}' "
                "with period='max'."
            )
 
        df = _clean_raw_df(raw)      # ascending, raw OHLCV only
        df = _build_derived(df)      # add all columns, sort desc, fill NaN
        df.to_csv(filepath, index=False)
 
        print(
            f"[OK] Created '{filepath}' — {len(df):,} rows "
            f"({df['Date'].iloc[-1]} → {df['Date'].iloc[0]})."
        )
        return df
 
    # ------------------------------------------------------------------
    # Branch B: file exists → incremental update
    # ------------------------------------------------------------------
    print(f"[INFO] '{filepath}' found. Checking for new data…")
 
    existing = pd.read_csv(filepath)
 
    if existing.empty or "Date" not in existing.columns:
        raise ValueError(
            f"'{filepath}' is empty or missing a 'Date' column. "
            "Delete it and re-run to rebuild from scratch."
        )
 
    last_date: datetime = pd.to_datetime(existing["Date"].max())
    start_date: str     = (last_date + timedelta(days=1)).strftime("%Y-%m-%d")
    today: str          = datetime.today().strftime("%Y-%m-%d")
 
    if start_date > today:
        print("[INFO] Data is already up-to-date. No new rows added.")
        return existing
 
    print(f"[INFO] Fetching data from {start_date} to {today}…")
    raw_new = yf.download(
        TICKER, start=start_date, end=today, interval="1d", progress=False
    )
 
    if raw_new.empty:
        print("[INFO] No new trading data returned by Yahoo Finance.")
        return existing
 
    new_rows = _clean_raw_df(raw_new)
    print(f"[INFO] {len(new_rows)} new row(s) fetched.")
 
    # Keep only raw OHLCV from existing data — derived cols will be recomputed
    base_cols = ["Date", "Open", "High", "Low", "Close", "Volume"]
    existing_base = existing[base_cols].copy()
    existing_base["Date"] = (
        pd.to_datetime(existing_base["Date"]).dt.strftime("%Y-%m-%d")
    )
 
    # Merge, deduplicate, sort ascending (required for diff/shift correctness)
    combined = pd.concat([existing_base, new_rows], ignore_index=True)
    combined.drop_duplicates(subset="Date", keep="last", inplace=True)
    combined.sort_values("Date", ascending=True, inplace=True)
    combined.reset_index(drop=True, inplace=True)
 
    # Recompute all derived columns across the full dataset
    combined = _build_derived(combined)
    combined.to_csv(filepath, index=False)
 
    print(
        f"[OK] '{filepath}' updated — {len(combined):,} total rows "
        f"({combined['Date'].iloc[-1]} → {combined['Date'].iloc[0]})."
    )
    return combined
 

In [7]:
# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------
 
if __name__ == "__main__":
    data = update_nifty_csv("../data/input/nifty_50.csv")
    print(f"Shape of nifty 50 data: Rows: {data.shape[0]}, columns: {data.shape[1]}")

[INFO] '../data/input/nifty_50.csv' not found. Downloading full NIFTY 50 history…
[OK] Created '../data/input/nifty_50.csv' — 4,554 rows (2007-09-17 → 2026-04-13).
Shape of nifty 50 data: Rows: 4554, columns: 12
